# PASSIM Baseline Retrieval: VD_BDC × VD_PSC

## 0. Config

In [1]:
!pip install pandas pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 51.2 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 MB 69.0 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python -m pip install --upgrade pip


In [2]:
import os
import json
import shutil
import subprocess
from pathlib import Path
from collections import defaultdict

import pandas as pd
import pyarrow.parquet as pq

# ── Paths ──────────────────────────────────────────────────────────────────
BDC_PATH          = Path("workspace/embeddings/VD_bdc_chunks.json")
PSC_PATH          = Path("workspace/embeddings/VD_psc_chunks.json")
GT_PATH           = Path("workspace/embeddings/ir_ground_truth.json")
OUTPUT_DIR        = Path("../evaluation/passim_baseline")
PASSIM_WORK_DIR   = Path("/dev/shm/passim_retrieval")   # RAM-backed

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── PASSIM parameters ──────────────────────────────────────────────────────
N          = 4
MIN_MATCH  = 3
GAP        = 50
MIN_ALIGN  = 8
MAX_DF     = 150   # relative to ~65k docs; raise to 200 if recall is 0

# ── Evaluation ─────────────────────────────────────────────────────────────
K_VALUES   = [5, 10, 20]
MODEL_NAME = "passim-baseline"

print("Config ready.")

Config ready.


## 1. Load corpora and ground truth

In [7]:
def load_chunks(path):
    with open(path) as f:
        data = json.load(f)
    chunks = data.get('chunks', data) if isinstance(data, dict) else data
    print(f"  {Path(path).name}: {len(chunks):,} chunks")
    return chunks

print("Loading corpora...")
bdc_chunks = load_chunks(BDC_PATH)
psc_chunks = load_chunks(PSC_PATH)

# Build char-length lookups (needed for normalisation)
bdc_char_len = {c['chunk_id']: len(c['text']) for c in bdc_chunks}
psc_char_len = {c['chunk_id']: len(c['text']) for c in psc_chunks}

# Load ground truth
with open(GT_PATH) as f:
    gt_raw = json.load(f)
ground_truth = gt_raw['ground_truth']   # list of dicts
gt_by_qid = {e['query_chunk_id']: set(e['relevant_chunks']) for e in ground_truth}

print(f"\nGround truth: {len(ground_truth)} entries")
print(f"  explicit: {sum(1 for e in ground_truth if e['reference_type']=='explicit')}")
print(f"  implicit: {sum(1 for e in ground_truth if e['reference_type']=='implicit')}")

Loading corpora...
  VD_bdc_chunks.json: 19,466 chunks
  VD_psc_chunks.json: 46,408 chunks

Ground truth: 100 entries
  explicit: 50
  implicit: 50


## 2. Prepare PASSIM input JSONL

In [4]:
# PASSIM input format: one JSON object per line
# Required fields: id (string), text (string)
# We add a 'corpus' field to enable filterpairs (BDC×PSC only)
#
# filterpairs expression:  corpus != corpus2
# This prevents BDC×BDC and PSC×PSC comparisons, keeping only cross-corpus pairs.

PASSIM_WORK_DIR.mkdir(parents=True, exist_ok=True)
input_path = PASSIM_WORK_DIR / "input.jsonl"

written = 0
with open(input_path, 'w', encoding='utf-8') as f:
    for c in bdc_chunks:
        text = c.get('text', '').strip()
        if not text:
            continue
        record = {'id': c['chunk_id'], 'text': text, 'corpus': 'bdc'}
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        written += 1
    for c in psc_chunks:
        text = c.get('text', '').strip()
        if not text:
            continue
        record = {'id': c['chunk_id'], 'text': text, 'corpus': 'psc'}
        f.write(json.dumps(record, ensure_ascii=False) + '\n')
        written += 1

print(f"Written {written:,} records to {input_path}")
print(f"  BDC: {len(bdc_chunks):,} | PSC: {len(psc_chunks):,}")

Written 65,874 records to /dev/shm/passim_retrieval/input.jsonl
  BDC: 19,466 | PSC: 46,408


## 3. Run PASSIM

In [5]:
import sys
# Install PASSIM
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/dasmiq/passim.git"],
    capture_output=True, text=True
)
print("PASSIM install:", "OK" if result.returncode == 0 else "FAILED")
if result.returncode != 0:
    print(result.stderr)

import shutil
passim_path = shutil.which("passim")
print(f"passim binary: {passim_path}")
assert passim_path, "passim not on PATH — run: export PATH=$PATH:~/.local/bin"

PASSIM install: OK
passim binary: /usr/local/bin/passim


In [6]:
import subprocess

# Install Java 17
r = subprocess.run(
    'apt-get update -q && apt-get install -y -q openjdk-17-jdk',
    shell=True, capture_output=True, text=True
)
print(r.stdout[-2000:])
print(r.stderr[-1000:] if r.returncode != 0 else "OK")

# Verify
r2 = subprocess.run('java -version', shell=True, capture_output=True, text=True)
print(r2.stderr)  # java -version prints to stderr

ng /usr/lib/jvm/java-17-openjdk-amd64/bin/jps to provide /usr/bin/jps (jps) in auto mode
update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/jrunscript to provide /usr/bin/jrunscript (jrunscript) in auto mode
update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/jshell to provide /usr/bin/jshell (jshell) in auto mode
update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/jstack to provide /usr/bin/jstack (jstack) in auto mode
update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/jstat to provide /usr/bin/jstat (jstat) in auto mode
update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/jstatd to provide /usr/bin/jstatd (jstatd) in auto mode
update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/serialver to provide /usr/bin/serialver (serialver) in auto mode
update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/jhsdb to provide /usr/bin/jhsdb (jhsdb) in auto mode
Setting up libgtk2.0-0:amd64 (2.24.33-2ubu

In [ ]:
import subprocess, os, shutil
from pathlib import Path

os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
Path("/tmp/spark_tmp").mkdir(exist_ok=True)

# Try u=2000 — memory held at u=500, so we have headroom
passim_out3 = PASSIM_WORK_DIR / "out_u5000"
if passim_out2.exists():
    shutil.rmtree(passim_out2)

env = os.environ.copy()
env["SPARK_SUBMIT_ARGS"] = (
    "--master local[*] "
    "--driver-memory 32g "
    "--executor-memory 8g "
    "--conf spark.sql.shuffle.partitions=200 "
    "--conf spark.default.parallelism=200 "
    "--conf spark.local.dir=/tmp/spark_tmp "
    "--conf spark.driver.extraJavaOptions=-Djava.io.tmpdir=/tmp/spark_tmp"
)

cmd = [
    "passim",
    "-n", str(N), "--min-match", str(MIN_MATCH),
    "-l", "2", "-u", "5000",
    "--gap", str(GAP), "--min-align", str(MIN_ALIGN),
    "--pairwise",
    "--fields", "corpus",
    "--filterpairs", "corpus <> corpus2",
    str(input_path), str(passim_out2),
]

print("Running with u=5000...")
result = subprocess.run(cmd, env=env)
print(f"Exit code: {result.returncode}")

Running with u=5000...


:: loading settings :: url = jar:file:/usr/local/lib/python3.11/dist-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.graphframes#graphframes-spark4_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7197c53e-0692-4fae-ae75-3e0f40fb1df4;1.0
	confs: [default]
	found io.graphframes#graphframes-spark4_2.13;0.9.3 in central
:: resolution report :: resolve 125ms :: artifacts dl 3ms
	:: modules in use:
	io.graphframes#graphframes-spark4_2.13;0.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1  

Namespace(id='id', text='text', locs='locs', pages='pages', minDF=2, maxDF=5000, min_match=3, n=4, floating_ngrams=False, complete_lines=False, gap=50, max_offset=20, beam=20, pcopy=0.8, min_align=8, src_overlap=0.9, dst_overlap=0.5, fields=['corpus'], filterpairs='corpus <> corpus2', all_pairs=True, pairwise=True, docwise=False, refpref=False, linewise=False, to_index=False, to_pairs=False, to_extents=False, link_model=None, link_features=None, log_level='WARN', shards=1, input_format='json', output_format='json', inputPath='/dev/shm/passim_retrieval/input.jsonl', outputPath='/dev/shm/passim_retrieval/out_u2000')
10:31:43.725 [Executor task launch worker for task 6.0 in stage 12.0 (TID 87)] ERROR org.apache.spark.memory.TaskMemoryManager - error while calling spill() on org.apache.spark.util.collection.unsafe.sort.UnsafeExternalSorter@3ba09b9e
java.io.IOException: No space left on device
	at java.base/java.io.FileOutputStream.writeBytes(Native Method) ~[?:?]
	at java.base/java.io.File

## 4. Parse PASSIM output → ranked results per BDC chunk

In [10]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
from collections import defaultdict
import json
from pathlib import Path

align_dir = passim_out2 / "align.json"
print(f"Reading from {align_dir}")
print(list(align_dir.iterdir())[:5])

records = []
for f in align_dir.glob("part-*.json"):
    with open(f) as fh:
        for line in fh:
            line = line.strip()
            if line:
                records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Total alignment records: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(df.head(3))
# Build BDC×PSC pairs — handle both directions
rows = []
for _, row in df.iterrows():
    id_a, corpus_a = row['id'], row['corpus']
    id_b, corpus_b = row['id2'], row['corpus2']
    # align_chars: use source-side span (end - begin)
    align_a = int(row['end']) - int(row['begin'])
    align_b = int(row['end2']) - int(row['begin2'])

    if corpus_a == 'bdc' and corpus_b == 'psc':
        rows.append({'bdc_id': id_a, 'psc_id': id_b, 'align_chars': align_a})
    elif corpus_a == 'psc' and corpus_b == 'bdc':
        rows.append({'bdc_id': id_b, 'psc_id': id_a, 'align_chars': align_b})

alignments = pd.DataFrame(rows) if rows else pd.DataFrame(columns=['bdc_id', 'psc_id', 'align_chars'])
print(f"\nBDC×PSC pairs: {len(alignments):,}")
print(f"Unique BDC chunks matched: {alignments['bdc_id'].nunique():,}")
print(f"Unique PSC chunks matched:  {alignments['psc_id'].nunique():,}")

# Score = align_chars / bdc chunk character length
alignments['bdc_char_len'] = alignments['bdc_id'].map(bdc_char_len).fillna(1)
alignments['score'] = alignments['align_chars'] / alignments['bdc_char_len']

# Deduplicate: keep max score per BDC-PSC pair (both directions may appear)
alignments = (
    alignments
    .groupby(['bdc_id', 'psc_id'], as_index=False)['score']
    .max()
    .sort_values(['bdc_id', 'score'], ascending=[True, False])
)

# Build retrieval_results: {bdc_chunk_id: [(psc_chunk_id, score), ...]}
retrieval_results = defaultdict(list)
for _, row in alignments.iterrows():
    retrieval_results[row['bdc_id']].append((row['psc_id'], row['score']))
for qid in retrieval_results:
    retrieval_results[qid].sort(key=lambda x: x[1], reverse=True)

print(f"\nretrieval_results keys (BDC chunks with ≥1 match): {len(retrieval_results):,}")
print(f"GT queries with any PASSIM match: {sum(1 for qid in gt_by_qid if qid in retrieval_results)} / {len(gt_by_qid)}")

if retrieval_results:
    all_scores = [s for hits in retrieval_results.values() for _, s in hits]
    print(f"Score stats — min={min(all_scores):.3f}  median={sorted(all_scores)[len(all_scores)//2]:.3f}  max={max(all_scores):.3f}")

Reading from /dev/shm/passim_retrieval/out_u2000/align.json
[PosixPath('/dev/shm/passim_retrieval/out_u2000/align.json/._SUCCESS.crc'), PosixPath('/dev/shm/passim_retrieval/out_u2000/align.json/_SUCCESS'), PosixPath('/dev/shm/passim_retrieval/out_u2000/align.json/.part-00002-2289976e-5bad-4815-9a2e-365eb2b6778b-c000.json.crc'), PosixPath('/dev/shm/passim_retrieval/out_u2000/align.json/part-00002-2289976e-5bad-4815-9a2e-365eb2b6778b-c000.json'), PosixPath('/dev/shm/passim_retrieval/out_u2000/align.json/.part-00005-2289976e-5bad-4815-9a2e-365eb2b6778b-c000.json.crc')]
Total alignment records: 229,098
Columns: ['uid2', 'uid', 'corpus', 'corpus2', 'begin2', 'end2', 'begin', 'end', 'shard', 'id', 'id2', 's1', 's2', 'matches']
                  uid2                  uid corpus corpus2  begin2  end2  \
0 -8607619769528989173 -9169811829278970603    bdc     psc     746   803   
1 -8476841340980987284 -9151214618220824697    bdc     psc     262   313   
2  9120955973315416860 -91512146182208246

In [14]:
summary, missed, hit_details = evaluate_passim(retrieval_results, ground_truth)

print("=" * 50)
print("PASSIM BASELINE  (n=4, m=3, g=50, a=8, u=2000)")
print("=" * 50)
for metric, val in summary.items():
    print(f"  {metric}: {val:.4f}")

print(f"\nQueries with no PASSIM match at all: {len(missed)} / {len(ground_truth)}")

for ref_type in ['explicit', 'implicit']:
    subset = [e for e in ground_truth if e['reference_type'] == ref_type]
    s, _, _ = evaluate_passim(retrieval_results, subset)
    print(f"\n{ref_type.capitalize()} (n={len(subset)}):")
    for metric, val in s.items():
        print(f"  {metric}: {val:.4f}")

PASSIM BASELINE  (n=4, m=3, g=50, a=8, u=2000)
  Recall@5: 0.4100
  MRR@5: 0.3350
  Recall@10: 0.4200
  MRR@10: 0.3361
  Recall@20: 0.4200
  MRR@20: 0.3361

Queries with no PASSIM match at all: 24 / 100

Explicit (n=50):
  Recall@5: 0.7400
  MRR@5: 0.6267
  Recall@10: 0.7400
  MRR@10: 0.6267
  Recall@20: 0.7400
  MRR@20: 0.6267

Implicit (n=50):
  Recall@5: 0.0800
  MRR@5: 0.0433
  Recall@10: 0.1000
  MRR@10: 0.0456
  Recall@20: 0.1000
  MRR@20: 0.0456


In [47]:
# Verify the chunk made it into the input file
import subprocess
r = subprocess.run(
    "grep -c '10041_sent_220_222' /dev/shm/passim_retrieval/input.jsonl",
    shell=True, capture_output=True, text=True
)
print(r.stdout)  # should be 1

1



In [ ]:
# Compute normalised score and deduplicate (keep best alignment per BDC-PSC pair)
alignments['bdc_char_len'] = alignments['bdc_id'].map(bdc_char_len).fillna(1)
alignments['score'] = alignments['align_chars'] / alignments['bdc_char_len']

# Deduplicate: if same BDC-PSC pair appears from both directions, keep max score
alignments = (
    alignments
    .groupby(['bdc_id', 'psc_id'], as_index=False)['score']
    .max()
    .sort_values(['bdc_id', 'score'], ascending=[True, False])
)

# Build retrieval_results dict: {bdc_chunk_id: [(psc_chunk_id, score), ...]}
retrieval_results = defaultdict(list)
for _, row in alignments.iterrows():
    retrieval_results[row['bdc_id']].append((row['psc_id'], row['score']))

# Sort each list by score descending (should already be sorted, but be safe)
for qid in retrieval_results:
    retrieval_results[qid].sort(key=lambda x: x[1], reverse=True)

print(f"retrieval_results keys (BDC chunks with ≥1 match): {len(retrieval_results):,}")
print(f"GT queries: {len(gt_by_qid):,}")
print(f"GT queries with any PASSIM match: {sum(1 for qid in gt_by_qid if qid in retrieval_results)}")
print()

# Score distribution
all_scores = [score for hits in retrieval_results.values() for _, score in hits]
print(f"Score stats across all pairs:")
print(f"  min={min(all_scores):.3f}  median={sorted(all_scores)[len(all_scores)//2]:.3f}  max={max(all_scores):.3f}")

## 5. Evaluate against ground truth (chunk level)

In [ ]:
# Evaluation mirrors the dense retrieval pipeline 

def evaluate_passim(retrieval_results, ground_truth_list, k_values=[5, 10, 20]):
    """
    Compute Recall@k and MRR@k for PASSIM retrieval.
    ground_truth_list: list of GT dicts (from ir_ground_truth.json['ground_truth'])
    retrieval_results: {query_chunk_id: [(psc_chunk_id, score), ...]}
    """
    metrics = {k: {'recall': [], 'mrr': []} for k in k_values}
    missed_queries = []   # GT queries with no PASSIM result at all
    hit_details   = []    # per-query hit info for inspection

    for entry in ground_truth_list:
        qid      = entry['query_chunk_id']
        gt_ids   = set(entry['relevant_chunks'])
        ref_type = entry['reference_type']

        if qid not in retrieval_results:
            missed_queries.append({'qid': qid, 'ref_type': ref_type, 'gt_ids': list(gt_ids)})
            for k in k_values:
                metrics[k]['recall'].append(0.0)
                metrics[k]['mrr'].append(0.0)
            continue

        retrieved = [cid for cid, _ in retrieval_results[qid]]
        hit_rank  = None

        for k in k_values:
            top_k = retrieved[:k]
            hit   = any(cid in gt_ids for cid in top_k)
            metrics[k]['recall'].append(1.0 if hit else 0.0)

            mrr = 0.0
            for rank, cid in enumerate(top_k, 1):
                if cid in gt_ids:
                    mrr = 1.0 / rank
                    if hit_rank is None:
                        hit_rank = rank
                    break
            metrics[k]['mrr'].append(mrr)

        hit_details.append({
            'qid': qid, 'ref_type': ref_type,
            'gt_ids': list(gt_ids),
            'hit_rank': hit_rank,
            'n_results': len(retrieved),
            'top5': retrieved[:5]
        })

    summary = {}
    for k in k_values:
        r = metrics[k]['recall']
        m = metrics[k]['mrr']
        summary[f'Recall@{k}'] = round(sum(r) / len(r), 4) if r else 0.0
        summary[f'MRR@{k}']    = round(sum(m) / len(m), 4) if m else 0.0

    return summary, missed_queries, hit_details


# ── Overall evaluation ──────────────────────────────────────────────────────
summary, missed, hit_details = evaluate_passim(retrieval_results, ground_truth)

print("=" * 50)
print(f"PASSIM BASELINE  ({MODEL_NAME})")
print("=" * 50)
for metric, val in summary.items():
    print(f"  {metric}: {val:.4f}")

print(f"\nQueries with no PASSIM match at all: {len(missed)} / {len(ground_truth)}")

# ── Stratified by reference type ────────────────────────────────────────────
for ref_type in ['explicit', 'implicit']:
    subset = [e for e in ground_truth if e['reference_type'] == ref_type]
    s, _, _ = evaluate_passim(retrieval_results, subset)
    print(f"\n{ref_type.capitalize()} references (n={len(subset)}):")
    for metric, val in s.items():
        print(f"  {metric}: {val:.4f}")

PASSIM BASELINE  (passim-baseline)
  Recall@5: 0.4100
  MRR@5: 0.3350
  Recall@10: 0.4200
  MRR@10: 0.3361
  Recall@20: 0.4200
  MRR@20: 0.3361

Queries with no PASSIM match at all: 24 / 100

Explicit references (n=50):
  Recall@5: 0.7400
  MRR@5: 0.6267
  Recall@10: 0.7400
  MRR@10: 0.6267
  Recall@20: 0.7400
  MRR@20: 0.6267

Implicit references (n=50):
  Recall@5: 0.0800
  MRR@5: 0.0433
  Recall@10: 0.1000
  MRR@10: 0.0456
  Recall@20: 0.1000
  MRR@20: 0.0456


In [68]:
summary, missed, hit_details = evaluate_passim(retrieval_results, ground_truth)

print("=" * 50)
print("PASSIM BASELINE  (n=4, m=3, g=50, a=8, u=500)")
print("=" * 50)
for metric, val in summary.items():
    print(f"  {metric}: {val:.4f}")

print(f"\nQueries with no PASSIM match at all: {len(missed)} / {len(ground_truth)}")

for ref_type in ['explicit', 'implicit']:
    subset = [e for e in ground_truth if e['reference_type'] == ref_type]
    s, _, _ = evaluate_passim(retrieval_results, subset)
    print(f"\n{ref_type.capitalize()} (n={len(subset)}):")
    for metric, val in s.items():
        print(f"  {metric}: {val:.4f}")

PASSIM BASELINE  (n=4, m=3, g=50, a=8, u=500)
  Recall@5: 0.1900
  MRR@5: 0.1850
  Recall@10: 0.1900
  MRR@10: 0.1850
  Recall@20: 0.1900
  MRR@20: 0.1850

Queries with no PASSIM match at all: 77 / 100

Explicit (n=50):
  Recall@5: 0.3800
  MRR@5: 0.3700
  Recall@10: 0.3800
  MRR@10: 0.3700
  Recall@20: 0.3800
  MRR@20: 0.3700

Implicit (n=50):
  Recall@5: 0.0000
  MRR@5: 0.0000
  Recall@10: 0.0000
  MRR@10: 0.0000
  Recall@20: 0.0000
  MRR@20: 0.0000


In [18]:
# Get the implicit hits with their rank and aligned text
implicit_hits = [h for h in hit_details if h['ref_type'] == 'implicit' and h['hit_rank'] is not None]
print(f"Implicit hits: {len(implicit_hits)}\n")

bdc_texts = {c['chunk_id']: c['text'] for c in bdc_chunks}
psc_texts = {c['chunk_id']: c['text'] for c in psc_chunks}

for h in implicit_hits:
    qid   = h['qid']
    gt_id = h['gt_ids'][0]
    
    # Find the actual PASSIM alignment row for this pair
    mask = (df['id'] == qid) & (df['id2'] == gt_id)
    if not mask.any():
        # try flipped direction
        mask = (df['id'] == gt_id) & (df['id2'] == qid)
    
    passim_row = df[mask]
    
    print(f"Rank {h['hit_rank']} | {qid}")
    print(f"  BDC: {bdc_texts.get(qid, '')[:200]}")
    print(f"  PSC: {gt_id, psc_texts.get(gt_id, '')[:200]}")
    if not passim_row.empty:
        print(f"  Aligned — s1: {passim_row.iloc[0]['s1']}")
        print(f"            s2: {passim_row.iloc[0]['s2']}")
        print(f"  matches={passim_row.iloc[0]['matches']}")
    print()

Implicit hits: 5

Rank 9 | 10936_sent_3
  BDC: Videor fortasse tibi , Bullingere , more andabatarum in tenebris ventilare gladium nec habere antagonistam , quem feriam aut quocum pugnem , cum , quod contendo et postulo , nemo inficietur .
  PSC: ('023_Hieronymus-Stridonensis_De-virginitate-B.-Mariae_window_19', 'coitum magis, quam ad scientiam referendum: quasi hoc quisquam negaverit, et eas ineptias quas redarguit, aliquando prudens quispiam potuerit suspicari. Deinde vult docere, quod donec, sive usque, adv')
  Aligned — s1: more andabatarum in tenebris ventilare gladium -n
            s2: more Andabatarum ----------------------gladium in
  matches=26

Rank 2 | 11754_sent_11
  BDC: Sin autem factu minus opus esse arbitrabimini , supprimatur , inanescat , flaccescat pulvereoque situ squaleat et pereat .
  PSC: ('004_Cyprianus-Carthaginensis_Ad-Demetrianum_window_12', 'Dominus (Amos IV, 7, 8) . VII. Indignatur ecce Dominus et irascitur, et quod ad eum non convertamini comminatur; et tu

In [30]:
import json
import pandas as pd
from pathlib import Path

# --- Ground truth ---
with open("../data/ir_ground_truth.json") as f:
    ground_truth_raw = json.load(f)
ground_truth = ground_truth_raw["ground_truth"]  # unwrap from metadata wrapper


# --- Retrieval results ---
with open("/Users/lenap/Desktop/Master-Thesis/bullinger-patristic-detection/retrieval/VD/locisimiles-e5-phil-v2/retrieval_results_locisimiles_e5-phil-v2.json") as f:
    retrieval_results = json.load(f)

# --- BDC / PSC chunks (adjust paths) ---
with open("../data/VD_bdc_chunks.json") as f:
    bdc_chunks = json.load(f)

with open("../data/VD_psc_chunks.json") as f:
    psc_chunks = json.load(f)


# --- Build hit_details from ground truth + retrieval results ---
# Assumes ground_truth is a list of {"qid":…, "gt_ids":[…], "ref_type":…}
# and retrieval_results is a list of {"qid":…, "results":[{"chunk_id":…}, …]}

retrieved_index = {
    qid: [c["candidate_id"] for c in candidates]
    for qid, candidates in retrieval_results["retrieval_results"].items()
}

hit_details = []
for entry in ground_truth:
    qid      = entry["query_chunk_id"]
    gt_ids   = entry["relevant_chunks"]
    ref_type = entry["reference_type"]
    ranked   = retrieved_index.get(qid, [])
    hit_rank = None
    for i, cid in enumerate(ranked, start=1):
        if cid in gt_ids:
            hit_rank = i
            break
    hit_details.append({
        "qid":      qid,
        "gt_ids":   gt_ids,
        "ref_type": ref_type,
        "hit_rank": hit_rank,
    })

# --- Implicit hits recovered by bi-encoder ---
implicit_hits = [
    h for h in hit_details
    if h["ref_type"] == "implicit"
    and h["hit_rank"] is not None
]
print(f"Implicit hits recovered by bi-encoder: {len(implicit_hits)}\n")

bdc_texts = {c["chunk_id"]: c["text"] for c in bdc_chunks["chunks"]}
psc_texts = {c["chunk_id"]: c["text"] for c in psc_chunks["chunks"]}

for h in implicit_hits:
    qid   = h["qid"]
    gt_id = h["gt_ids"][0]

    print(f"Rank {h['hit_rank']:>3d} | {qid}")
    print(f"  BDC : {bdc_texts.get(qid, '[not found]')[:1000]}")
    print(f"  PSC : {psc_texts.get(gt_id, '[not found]')[:1000]}")
    print(f"  GT  : {gt_id}")

Implicit hits recovered by bi-encoder: 7

Rank  19 | 10501_sent_13
  BDC : «Sacramentum est invisibilis gratiae visibile signum . »
  PSC : Adamum intravit, liberaremur. His dictis, interrogandus est an haec credat atque observare desideret. Quod cum responderit, solemniter utique signandus est et Ecclesiae more tractandus. De sacramento sane quod accipit, cum ei bene commendatum fuerit, signacula quidem rerum divinarum esse visibilia, sed res ipsas invisibiles in eis honorari; nec sic habendam esse illam speciem benedictione sanctificatam, quemadmodum habetur in usu quolibet: dicendum etiam quid significet et sermo ille quem audivit, quid in illo condiat, cujus illa res similitudinem gerit. Deinde monendus est ex hac occasione, ut si quid etiam in Scripturis audiat quod carnaliter sonet, etiamsi non intelligit, credat tamen spirituale aliquid significari, quod ad sanctos mores futuramque vitam pertineat. Hoc autem ita breviter discit, ut quidquid audierit ex Libris canonicis quod ad d